# Setting up the stage

In this first notebook we will introduce the key concepts of reinforcement learning. 

<div class="alert alert-success">

**Learning outcomes:**  
By the end of this notebook you should be able to define the key vocabulary and ingredients of reinforcement learning:
- Markov Decision Processes,
- trajectories and samples,
- policies,
- value functions.

And give a general definition of RL.
</div>

Along the way, we will play with a few examples.

# In plain words

<div class="alert alert-success">
    
Reinforcement Learning is about learning an optimal sequential behavior for a given dynamical system.
</div>

Let's break this down.
- dynamical system  
$\rightarrow$ a set of *state* variables that evolve through time, under the influence of *decision variables*.
- sequential behavior   
$\rightarrow$ discrete time steps, sequence of decisions
- optimal  
$\rightarrow$ a reward signal informs us of the quality of the last action
- learning  
$\rightarrow$ no known model a priori, just interaction samples, behavior adaptation.

<center><img src="img/dynamic.png" style="width: 400px;"></img></center>

<div class="alert alert-success">

**Keywords:**
- system to control / environment
- control policy
- optimality
</div>
Inspirations for RL:
- Control theory and Stochastic processes for the **modeling** part
- Statistics, Optimization and Cognitive Psychology for the **learning** part

In this notebook, we will introduce and illustrate the concepts needed to define the RL problem.

# From plain words to first variables

## A medical prescription example

<center><img src="img/patient-doctor.png" style="height: 200px;"></center>
    
A patient walks into a clinic with their medical file (medical history, x-rays, blood work, etc.). You, as their doctor, need to write a prescription. Let us use this example to formalize the process of deciding what to write on the prescription.

## Patient variables

<center>
<img src="img/patient_file.png" style="height: 100px;"> </img> <br>
Patient state now: $S_0$  <br>
Future states: $S_t$
</center>

The medical file of the patient allows us to define a number of variables that characterize the patient now. We will write $S_0$ the vector of these variables. Future measurements will be noted $S_t$.

$S_t$ is a random vector, taking different values in a *patient description space* $\mathcal{S}$ at different time steps.

## Prescription

<center>
<img src="img/prescription.png" style="height: 100px;"> </img> <br>
Prescription: $\left( A_t \right)_{t\in\mathbb{N}} = (A_0, A_1, A_2, ...)$
</center>

The prescription is a series of recommendations we give to the patient over the course of treatment. It is thus a sequence $\left( A_t \right)_{t\in\mathbb{N}} = (A_0, A_1, A_2, ...)$ of variables $A_t$.

These treatments $A_t$ are random variables too, taking their value in some space $\mathcal{A}$.

## Patient evolution


<center>
<img src="img/patient_evolution.png" style="height: 100px;"> </img> <br>
    $\mathbb{P}(S_t)$?
</center>

The patient evolves over time steps. Their evolution follows a certain probability distribution $\mathbb{P}(S_t)$ over descriptive states.

So $\left( S_t \right)_{t\in\mathbb{N}}$ defines a *random process* that describes the patient's evolution under the influence of past $S_t$ and $A_t$.

## Physician's goal

<center><img src="img/patient_happy.png" style="height: 100px;"> </img> <br></center>

$$J \left( \left(S_t\right)_{t\in \mathbb{N}}, \left( A_t \right)_{t\in \mathbb{N}} \right)?$$

The physician's goal is to bring the patient from an unhealthy state $S_0$ to a healthy situation.  

This goal is not only defined by a final state of the patient but by the full trajectory followed by the variables $S_t$ and $A_t$. For example, prescribing a drug that damages the patient's liver, or letting the patient experience too much pain over the course of treatment is discouraged.

We define a criterion $J \left( \left(S_t\right)_{t\in \mathbb{N}}, \left( A_t \right)_{t\in \mathbb{N}} \right)$ that allows to quantify how good a trajectory in the joint $\mathcal{S}\times \mathcal{A}$ space is.

## Wrap-up

- Patient state $S_t$, random variable,
- Physician instruction $A_t$, random variable,
- Prescription $\left( A_t \right)_{t\in\mathbb{N}}$, sequence of random variables, random process  
- Patient's evolution $\mathbb{P}(S_t)$,  
- Patient's state trajectory $\left( S_t \right)_{t\in\mathbb{N}}$, random process, 
- Patient's full trajectory $\left( S_t, A_t \right)_{t\in\mathbb{N}}$, random process, 
- Value of a trajectory $J \left( \left(S_t, A_t \right)_{t\in \mathbb{N}} \right)$.  

It seems reasonable that the physician's recommendation $\mathbb{P}(A_t)$ at step $t$ be dependent on previously observed states $\left(S_0, \ldots, S_t\right)$ and recommended treatments $\left(A_0, \ldots, A_{t-1}\right)$.

# Common misconception

You will often see the following type of drawing, along with a sentence like "RL is concerned with the problem on an agent performing actions to control an environment". 

<center><img src="img/misconception.png" style="height: 300px;"></img></center>

Although this sentence is not false *per se*, it conveys an important misconception that may be grounded in too simple anthropomorphic analogies. One often talks about the *state of the agent* or the *state of the environment*. The distinction here is confusing at best: there is no separation between agent and environment. A better vocabulary is to talk about a *system to control*, that is described through its observed *state*. This system is controlled by the application of actions issued from a *policy* or *control law*. The process of *learning* this policy is what RL is concerned with.

Although less shiny, the drawing below may be less misleading.

<center><img src="img/dynamic.png" style="height: 300px;"></img></center>

# Markov Decision Processes

At every time step, the system state is $S_t$ and we decide to apply action $A_t$. This results in observing a new state $S_{t+1}$ and receiving a scalar reward signal $R_t$ for this transition.

$R_t$ tells us how happy we are with the last transition.

We will now make our main assumption about the systems we want to control.

<div class="alert alert-success">
    
**Fundamental assumption (Markov property)**
$$\mathbb{P}(S_{t+1},R_t|S_t, A_t, S_{t-1}, A_{t-1}, \ldots, S_0, A_0) = \mathbb{P}(S_{t+1},R_t|S_t, A_t)$$
</div>
    
Such a system will be called a Markov Decision Process (MDP).

One generally separates the state dynamics and the rewards by:
$$\mathbb{P}(S_{t+1},R_t|S_t, A_t) = \mathbb{P}(S_{t+1}|S_t, A_t)\cdot \mathbb{P}(R_t|S_t, A_t, S_{t+1})$$

Which leads in turn to the general definition of an MDP:
<div class="alert alert-success"><b>Markov Decision Process (MDP)</b><br>
A Markov Decision Process is given by:
<ul>
<li> A set of states $\mathcal{S}$
<li> A set of actions $\mathcal{A}$
<li> A (Markovian) transition model $\mathbb{P}\left(S_{t+1} | S_t, A_t \right)$, noted $p_t(s'|s,a)$
<li> A reward model $\mathbb{P}\left( R_t | S_t, A_t, S_{t+1} \right)$, noted $r_t(s,a)$ or $r_t(s,a,s')$
<li> A set of discrete decision epochs $\mathcal{T}=\{0,1,\ldots,h\}$
</ul>
</div>

Most of the results presented here can be found in M. L. Puterman's classic book, **[Markov Decision Processes: Discrete Stochastic Dynamic Programming](https://www.wiley.com/en-us/Markov+Decision+Processes%3A+Discrete+Stochastic+Dynamic+Programming-p-9781118625873)**.

If $h\rightarrow\infty$ we have an infinite horizon control problem.

<div class="alert alert-success">

Since we will work most of the time with stationary, infinite horizon problems, we shall identify the MDP with the 4-tuple $(\mathcal{S},\mathcal{A},p,r)$.    
</div>


In the general case, $\mathcal{S}$ and $\mathcal{A}$ may each be either:
  - arbitrary finite sets,
  - arbitrary countable infinite sets,
  - compact subsets of a finite dimensional Euclidean space, or
  - non-empty Borel subsets of complete, separable metric spaces.

The reward model can be written interchangeably $r(s,a)$ or $r(s,a,s')$, depending on authors, with $r(s,a) = \mathbb{E}_{s'\sim p(\cdot|s,a)} [r(s,a,s')]$. We will use both notations in the rest of the class.

So, in RL, we wish to control the trajectory of a system that, we suppose, behaves as a Markov Decision Process.

<center><img src="img/dynamic.png" style="height: 240px;"></img></center>

# Back to the V2G problem

<div class="alert alert-warning">

**Discussion**  
Is this a Markov Decision Process?  
What is the state space? The action space? The transition dynamics?  
Is the process stationary? What is the horizon?
</div>